## ML Assignment 2
Name: Thosar Keshav Jay Radha   
BITS ID: 2026AJ05216

In [42]:
!pip install scikit-learn imbalanced-learn

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [44]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

In [46]:
file_path = '/home/s2026aj05216/Downloads/IT_customer_churn.csv'
df = pd.read_csv(file_path)

print(f"Dataset has {df.shape[0]} rows, {df.shape[1]} columns")

Dataset has 7043 rows, 20 columns


In [47]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [49]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [50]:
df.isnull().sum()

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [51]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

Numerical Columns Summary

In [53]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.794470
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,398.550000
50%,0.000000,29.000000,70.350000,1394.550000
75%,0.000000,55.000000,89.850000,3786.600000
max,1.000000,72.000000,118.750000,8684.800000


Categorical Colummns Summary

In [55]:
df.describe(include=['O', 'object', 'category'])

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,2
top,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,No
freq,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,5174


Missing Values

In [57]:
categorical_data = ['gender', 'Partner', 'Dependents',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

for col in categorical_data:
    print(df[col].value_counts())
    print()

gender
Male      3555
Female    3488
Name: count, dtype: int64

Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

TechSupport
No                     3473
Yes                    2044
No internet service    1526
Name: count, dtype: int64

StreamingTV
No                 

In [58]:
df['Churn'].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

The target class is highly imbalanced, hence we need to use SMOTE for generate samples for the positive churn cases. Before that we will apply some preprocessing to the dataset

In [60]:
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = categorical_data.copy()

scaler = StandardScaler()
X_num_scaled = pd.DataFrame(scaler.fit_transform(X[numeric_features]), columns=numeric_features)

encoder = OneHotEncoder(drop='first', sparse_output=False)
X_cat_encoded = pd.DataFrame(encoder.fit_transform(X[categorical_features]), columns=encoder.get_feature_names_out(categorical_features))

X_processed = pd.concat([X_num_scaled, X_cat_encoded], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [61]:
print("Original training shape:", X_train.shape, y_train.shape)
print("Resampled training shape (after SMOTE):", X_train_resampled.shape, y_train_resampled.shape)

Original training shape: (5634, 29) (5634,)
Resampled training shape (after SMOTE): (8278, 29) (8278,)


In [62]:
y_train_resampled.value_counts()

Churn
0    4139
1    4139
Name: count, dtype: int64

In [63]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'KNeighborsClassifier': KNeighborsClassifier(),
    'MultinomialNB': MultinomialNB(),
    'SVC': SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    if name == 'MultinomialNB':
        # since MultinomialNB cannot handle negative values and we have user StandardScaler earlier
        mm_scaler = MinMaxScaler()
        X_train_res_nb = mm_scaler.fit_transform(X_train_resampled)
        X_test_nb = mm_scaler.transform(X_test)

        model.fit(X_train_res_nb, y_train_resampled)

        y_pred = model.predict(X_test_nb)
        y_prob = model.predict_proba(X_test_nb)[:, 1]
    else:
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'AUC': auc,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'MCC': mcc
    })

/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [64]:
results_df = pd.DataFrame(results)
results_df.style.highlight_max(axis=0, color='lightgreen', subset=['Accuracy', 'AUC', 'Precision', 'Recall', 'F1 Score', 'MCC'])

,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,LogisticRegression,0.735983,0.838950,0.501678,0.799465,0.616495,0.458077
1,DecisionTreeClassifier,0.716111,0.664783,0.470455,0.553476,0.508600,0.312868
2,KNeighborsClassifier,0.699787,0.764405,0.458545,0.724599,0.561658,0.371726
3,MultinomialNB,0.652236,0.789194,0.421622,0.834225,0.560144,0.372004
4,SVC,0.757275,0.824521,0.530189,0.751337,0.621681,0.465573


In [65]:
import joblib
import os

# Create a directory to store models if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save each model to a joblib file
for name, model in models.items():
    file_path = f'models/{name}.joblib'
    joblib.dump(model, file_path)
    print(f"Successfully saved {name} to {file_path}")

Successfully saved LogisticRegression to models/LogisticRegression.joblib
Successfully saved DecisionTreeClassifier to models/DecisionTreeClassifier.joblib
Successfully saved KNeighborsClassifier to models/KNeighborsClassifier.joblib
Successfully saved MultinomialNB to models/MultinomialNB.joblib
Successfully saved SVC to models/SVC.joblib
